# Multiple Disease Prediction System - Parkinson's Disease Prediction
This notebook implements a subject-aware machine learning workflow using a `Pipeline(StandardScaler + SVC(kernel='linear'))` to predict Parkinson's disease from biomedical voice measurements while strictly preventing subject data leakage.

### Importing the Dependencies

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedGroupKFold
from sklearn import svm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pickle
import joblib

### Data Collection & Analysis

In [ ]:
# Loading the Parkinson's dataset
parkinsons_data = pd.read_csv('../Dataset/parkinsons.csv')

In [ ]:
# Display first 5 rows
parkinsons_data.head()

In [ ]:
# Dataset shape
parkinsons_data.shape

In [ ]:
# Dataset information
parkinsons_data.info()

In [ ]:
# Check for missing values
parkinsons_data.isnull().sum()

In [ ]:
# Statistical summary
parkinsons_data.describe()

In [ ]:
# Target distribution (1 --> Parkinson's Positive, 0 --> Healthy)
parkinsons_data['status'].value_counts()

In [ ]:
# Mean feature values grouped by disease status
parkinsons_data.groupby('status').mean(numeric_only=True)

### Subject Extraction & Group Distribution Analysis
The `name` column contains patient identifiers formatted as `phon_R01_S<ID>_<recording>`. Each subject has multiple voice recordings (6-7 replicates). Extracting subject IDs is essential to prevent subject leakage across train/test splits.

In [ ]:
# Extract Subject ID from the 'name' column
parkinsons_data['subject'] = parkinsons_data['name'].apply(lambda x: x.split('_')[2])

subject_summary = parkinsons_data.groupby('subject')['status'].agg(recordings='count', status='first')
print(f"Total voice recordings: {len(parkinsons_data)}")
print(f"Total unique subjects: {parkinsons_data['subject'].nunique()}")
print(f"\nUnique subjects per class:\n{subject_summary['status'].value_counts()}")

### Data Pre-Processing & Subject-Aware Splitting

In [ ]:
# Separating features (X), target (Y), and subject groups
X = parkinsons_data.drop(columns=['name', 'status', 'subject'])
Y = parkinsons_data['status']
groups = parkinsons_data['subject']

In [ ]:
# Subject-level Stratified Train/Test Split (80% train, 20% test)
# Ensures recordings from any given subject appear strictly in train OR test, never both
subject_df = parkinsons_data[['subject', 'status']].drop_duplicates()
train_subjects, test_subjects = train_test_split(
    subject_df['subject'],
    test_size=0.2,
    stratify=subject_df['status'],
    random_state=2
)

train_mask = parkinsons_data['subject'].isin(train_subjects)
test_mask = parkinsons_data['subject'].isin(test_subjects)

X_train, Y_train = X[train_mask], Y[train_mask]
X_test, Y_test = X[test_mask], Y[test_mask]

print(f"Total recordings: {X.shape[0]} across {subject_df.shape[0]} subjects")
print(f"Training set: {X_train.shape[0]} recordings across {len(train_subjects)} subjects (Status: {Y_train.value_counts().to_dict()})")
print(f"Test set: {X_test.shape[0]} recordings across {len(test_subjects)} subjects (Status: {Y_test.value_counts().to_dict()})")

### Model Training: Pipeline (StandardScaler + Linear SVC)
Support Vector Machines depend heavily on feature scaling because differences in scale distort the margin. Using a `Pipeline` ensures `StandardScaler` learns statistics strictly from the training data, preventing data leakage during preprocessing.

In [ ]:
# Create Pipeline: StandardScaler -> Linear SVC
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', svm.SVC(kernel='linear'))
])

print("Pipeline created:")
print(pipeline)

In [ ]:
# Fit Pipeline on training data only
pipeline.fit(X_train, Y_train)
print("Pipeline successfully fitted on training data.")

### Model Evaluation

In [ ]:
# Evaluation on Training Data
X_train_prediction = pipeline.predict(X_train)
training_data_accuracy = accuracy_score(Y_train, X_train_prediction)
print(f'Accuracy score of training data : {training_data_accuracy*100:.2f}%')

In [ ]:
# Comprehensive Model Evaluation on Test Data (Unseen Subjects)
X_test_prediction = pipeline.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)

print(f'Accuracy score of test data : {test_data_accuracy*100:.2f}%')

# Confusion Matrix
cm = confusion_matrix(Y_test, X_test_prediction)
print('\nConfusion Matrix:')
print(cm)
print(f'  True Negatives: {cm[0,0]}, False Positives: {cm[0,1]}')
print(f'  False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}')

# Classification Report
print('\nClassification Report:')
print(classification_report(Y_test, X_test_prediction, target_names=['Healthy (0)', "Parkinson's (1)"]))

# Individual Metrics
precision = precision_score(Y_test, X_test_prediction)
recall = recall_score(Y_test, X_test_prediction)
f1 = f1_score(Y_test, X_test_prediction)
y_scores = pipeline.decision_function(X_test)
roc_auc = roc_auc_score(Y_test, y_scores)

print(f'Precision: {precision:.4f}')
print(f'Recall (Sensitivity): {recall:.4f}')
print(f'F1-Score: {f1:.4f}')
print(f'ROC-AUC Score: {roc_auc:.4f}')

### Subject-Aware Cross-Validation
To rigorously evaluate generalization without subject leakage across the entire dataset, we use **5-Fold `StratifiedGroupKFold`** grouped by `subject`. Each fold evaluates on completely unseen subjects while preserving class distribution.

In [ ]:
# 5-Fold Stratified Group Cross-Validation on the full dataset using Pipeline
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2)

cv_acc = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='accuracy')
cv_precision = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='precision')
cv_recall = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='recall')
cv_f1 = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='f1')
cv_roc = cross_val_score(pipeline, X, Y, cv=sgkf, groups=groups, scoring='roc_auc')

print('=== 5-Fold Stratified Group Cross-Validation Results ===')
print(f'Accuracy : {cv_acc.mean()*100:.2f}% (+/- {cv_acc.std()*100:.2f}%)')
print(f'Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std():.4f})')
print(f'Recall   : {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})')
print(f'F1-Score : {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')
print(f'ROC-AUC  : {cv_roc.mean():.4f} (+/- {cv_roc.std():.4f})')

### Building a Predictive System

In [ ]:
# Sample input data (22 biomedical voice features)
input_data = (197.07600, 206.89600, 192.05500, 0.00289, 0.00001, 0.00166, 0.00168, 0.00498, 0.01098, 0.09700, 0.00563, 0.00680, 0.00802, 0.01689, 0.00339, 26.77500, 0.422229, 0.741367, -7.348300, 0.177551, 1.743867, 0.085569)

# Convert to DataFrame with feature column names
input_df = pd.DataFrame([input_data], columns=X.columns)

prediction = pipeline.predict(input_df)
print(f"Prediction: {prediction}")

if prediction[0] == 0:
    print("The person does not have Parkinson's disease")
else:
    print("The person has Parkinson's disease")

### Saving the Trained Pipeline

In [ ]:
# Saving the complete Pipeline (scaler + classifier)
filename = '../saved_models/parkinsons_pipeline.joblib'
joblib.dump(pipeline, filename)
print(f'Pipeline saved to {filename}')